In [0]:
%run /Workspace/Users/antoniorad15@gmail.com/ROBOTICS-AI-training-pipeline/pipeline-finetune-gr00t/secrets-template

In [0]:
%python
import os

ssh_priv_key = dbutils.secrets.get(scope="brev", key="ssh_private_key")

with open("/tmp/ssh_private_key_3", "w") as f:
    f.write(ssh_priv_key + "\n")
os.chmod("/tmp/ssh_private_key_3", 0o600)

In [0]:
%sh
ssh -i /tmp/ssh_private_key_3 -o StrictHostKeyChecking=no shadeform@$BREV_IP << 'EOF'
echo == OS ==
. /etc/os-release
echo $PRETTY_NAME
echo == GPU / Driver ==
nvidia-smi
EOF

In [0]:
%sh
scp -i /tmp/ssh_private_key_3 -o StrictHostKeyChecking=no \
  /Volumes/workspace/default/hdf52lerobot_script_files_metrics/merge_lerobot_datasets.py \
  shadeform@$BREV_IP:~/scripts/merge_lerobot_datasets.py

In [0]:
%sh
ssh -i /tmp/ssh_private_key_3 -o StrictHostKeyChecking=no shadeform@$BREV_IP << 'EOF'
tmux kill-session -t merge 2>/dev/null
tmux new-session -d -s merge
tmux send-keys -t merge "python3 ~/scripts/merge_lerobot_datasets.py 2>&1 | tee ~/scripts/logs/merge_lerobot_datasets.log" Enter
tmux send-keys -t merge "exit" Enter
EOF

In [0]:
%sh
echo "Waiting for data conversion to complete..."
while true; do
  echo "$(date): Checking status..."
  
  OUTPUT=$(echo '
    echo "=== TMUX SESSIONS ==="
    tmux list-sessions 2>&1 || echo "NO_SESSIONS"
    echo "=== CHECKING FINETUNE ==="
    if tmux has-session -t merge 2>/dev/null; then
      echo "STATUS_RUNNING"
    else
      echo "STATUS_DONE"
    fi
  ' | ssh -i /tmp/ssh_private_key_3 -o StrictHostKeyChecking=no shadeform@$BREV_IP 2>&1)
  
  EXIT_CODE=$?
  
  echo "--- SSH Exit Code: $EXIT_CODE ---"
  echo "--- Full Output ---"
  echo "$OUTPUT"
  echo "-------------------"
  
  if [ $EXIT_CODE -ne 0 ]; then
    echo "WARNING: SSH command failed!"
  fi
  
  if echo "$OUTPUT" | grep -q "STATUS_DONE"; then
    echo "Data conversion complete!"
    break
  fi
  
  sleep 60
done

ssh -i /tmp/ssh_private_key_3 -o StrictHostKeyChecking=no shadeform@$BREV_IP << EOF
cd \$HOME/lerobot_datasets/
du -sh .
ls -lh
EOF

In [0]:
%sh
rsync -avz --progress \
  -e "ssh -i /tmp/ssh_private_key_3 -o StrictHostKeyChecking=no" \
  shadeform@$BREV_IP:~/lerobot_datasets/merged_dataset/ \
  /Volumes/workspace/default/finetune_lerobot_datasets/merged_dataset/

In [0]:
import requests
import os
NEXT_JOB_ID = 449953930494301  

response = requests.post(
    f"{os.environ['DATABRICKS_HOST']}/api/2.1/jobs/run-now",
    headers={"Authorization": f"Bearer {os.environ['DATABRICKS_TOKEN']}"},
    json={"job_id": NEXT_JOB_ID}
)

if response.status_code == 200:
    print(f"Job 2 triggered: run_id={response.json()['run_id']}")
else:
    raise Exception(f"Failed to trigger Job 2: {response.text}")